<a href="https://colab.research.google.com/github/Sivabhargav14/FireRiskPredictionSystem/blob/main/FireRiskPredictionSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from IPython.display import clear_output
!pip install gtts

In [ ]:
df=pd.read_csv("/content/Trainingdataset.csv")

In [ ]:
print(df.head())
print(df.columns)
print(df.info())

   S.No.         UTC  Temperature[C]  Humidity[%]  TVOC[ppb]  eCO2[ppm]  \
0      0  1654733331          20.000        57.36          0        400   
1      1  1654733332          20.015        56.67          0        400   
2      2  1654733333          20.029        55.96          0        400   
3      3  1654733334          20.044        55.28          0        400   
4      4  1654733335          20.059        54.69          0        400   

   Raw H2  Raw Ethanol  Pressure[hPa]  PM1.0  PM2.5  NC0.5  NC1.0  NC2.5  CNT  \
0   12306        18520        939.735    0.0    0.0    0.0    0.0    0.0    0   
1   12345        18651        939.744    0.0    0.0    0.0    0.0    0.0    1   
2   12374        18764        939.738    0.0    0.0    0.0    0.0    0.0    2   
3   12390        18849        939.736    0.0    0.0    0.0    0.0    0.0    3   
4   12403        18921        939.744    0.0    0.0    0.0    0.0    0.0    4   

   Fire Alarm  
0           0  
1           0  
2           0 

In [ ]:
x = df.drop(columns=["Fire Alarm", "CNT", "UTC", "S.No."], errors='ignore')
y = df["Fire Alarm"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
x_train = x_train.values
x_test = x_test.values

In [ ]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=300,max_depth=20,class_weight='balanced',random_state=42)
model.fit(x_train_scaled, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=20, n_estimators=300,
                       random_state=42)

In [ ]:
y_prediction = model.predict(x_test_scaled)       #No fire:0/Fire:1
y_probability = model.predict_proba(x_test_scaled)[:, 1]  #Probability of fire:Low risk/High risk

In [ ]:
accuracy=accuracy_score(y_test, y_prediction)
roc=roc_auc_score(y_test, y_probability)
precision = precision_score(y_test, y_prediction)
recall = recall_score(y_test, y_prediction)
f1 = f1_score(y_test, y_prediction)

In [ ]:
print("Accuracy:", accuracy)
print("ROC-AUC:", roc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 1.0
ROC-AUC: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


In [ ]:
print(x.columns)

Index(['Temperature[C]', 'Humidity[%]', 'TVOC[ppb]', 'eCO2[ppm]', 'Raw H2',
       'Raw Ethanol', 'Pressure[hPa]', 'PM1.0', 'PM2.5', 'NC0.5', 'NC1.0',
       'NC2.5'],
      dtype='object')


In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
temp = widgets.FloatText(description="Temp (°C)")
humidity = widgets.FloatText(description="Humidity (%)")
tvoc = widgets.FloatText(description="TVOC")
eco2 = widgets.FloatText(description="eCO2")
h2 = widgets.FloatText(description="H2")
ethanol = widgets.FloatText(description="Ethanol")
pressure = widgets.FloatText(description="Pressure")
pm1 = widgets.FloatText(description="PM1.0")
pm25 = widgets.FloatText(description="PM2.5")
nc05 = widgets.FloatText(description="NC0.5")
nc1 = widgets.FloatText(description="NC1.0")
nc25 = widgets.FloatText(description="NC2.5")

In [ ]:
button = widgets.Button(description="Predict Fire Risk")
output = widgets.Output()

In [ ]:
def on_button_click(b):
    with output:
        output.clear_output()

        try:
            values = [
    temp.value, humidity.value, tvoc.value, eco2.value,
    h2.value, ethanol.value, pressure.value,
    pm1.value, pm25.value, nc05.value, nc1.value, nc25.value
]
            data = np.array([values])
            data = scaler.transform(data)

            risk = model.predict_proba(data)[0][1]

            if risk > 0.8:
                level = "EXTREME RISK"
            elif risk > 0.6:
                level = "HIGH RISK"
            elif risk > 0.3:
                level = "MEDIUM RISK"
            else:
                level = "LOW RISK"

            print("Fire Risk:", round(risk, 2))
            print("Level:", level)

            from gtts import gTTS
            from IPython.display import Audio

            text = f"Fire risk is {round(risk,2)}. Level is {level}"

            tts = gTTS(text=text, lang='en')
            tts.save("output.mp3")

            display(Audio("output.mp3", autoplay=True))

        except Exception as e:
            print("Error:", e)

In [ ]:
button.on_click(on_button_click)

In [ ]:
ui = widgets.VBox([
    temp, humidity, tvoc, eco2, h2, ethanol,
    pressure, pm1, pm25, nc05, nc1, nc25,
    button,
    output
])
display(ui)